<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2 align="center"> Trabalho 01</h2>
<h3 align="center"> José Diniz - RA: 183134</h3>

<h3 align="center"> Roberto Rodrigues - RA: 060235</h3>

Um sistema de classificação de tópicos consiste em um modelo capaz de identificar a qual domínio uma determinada sentença ou texto pertence. 

Crie um classificador que receba uma frase em inglês e indique se ela faz parte de uma das seguintes categorias: aventura, ficção científica, religião ou governo, utilizando o corpus em anexo.
Use as etapas de pré-processamento que vimos em sala e representações de texto diferentes se desejar (bag-of-words/LSA, Skip-gram, CBOW, GloVe). 

Utilize os algoritmos de classificação que desejar, e faça seleção de hiperparâmetros. Siga sempre as boas práticas para experimentos de aprendizado de máquina para evitar underfitting e overfitting.

O grupo que obtiver o melhor resultado no corpus de teste (de posse somente do professor) receberá 1 ponto a mais na média de disciplina. Se mais de um grupo tiver o mesmo melhor resultado, nenhum grupo ganhará essa pontuação.

A entrega deve ser feita por meio de script no kaggle compartilhado com o usuário do professor. Basta copiar o link na submissão da solução.

In [100]:

import logging
import pandas as pd
import numpy as np
from numpy import random
#import gensim
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
import re
from bs4 import BeautifulSoup
%matplotlib inline


<h3>1. Lendo o data set</h3>

In [101]:
df = pd.read_csv('corpus_categorias_treino.csv')
df.head(10)

,Unnamed: 0,words,category
0,0,He certainly didn't want a wife who was fickle...,adventure
1,1,"If he had married her , he'd have been asking ...",adventure
2,2,Sometimes he woke up in the middle of the nigh...,adventure
3,3,His plans and dreams had revolved around her s...,adventure
4,4,The best antidote for the bitterness and disap...,adventure
5,5,He found that if he was tired enough at night ...,adventure
6,6,Each day he found himself thinking less often ...,adventure
7,7,"each day the hurt was a little duller , a litt...",adventure
8,8,He had plenty of work to do .,adventure
9,9,"The grass in the meadows came fast , now that ...",adventure


In [77]:
df = pd.read_csv('corpus_categorias_treino.csv')
df = df[pd.notnull(df['category'])]
print(df.head(10))
print(df['words'].apply(lambda x: len(x.split(' '))).sum())

   Unnamed: 0                                              words   category
0           0  He certainly didn't want a wife who was fickle...  adventure
1           1  If he had married her , he'd have been asking ...  adventure
2           2  Sometimes he woke up in the middle of the nigh...  adventure
3           3  His plans and dreams had revolved around her s...  adventure
4           4  The best antidote for the bitterness and disap...  adventure
5           5  He found that if he was tired enough at night ...  adventure
6           6  Each day he found himself thinking less often ...  adventure
7           7  each day the hurt was a little duller , a litt...  adventure
8           8                      He had plenty of work to do .  adventure
9           9  The grass in the meadows came fast , now that ...  adventure
153347


In [102]:
X = df.words
y = df.category
categories = df.category.unique()

In [103]:
X

0       He certainly didn't want a wife who was fickle...
1       If he had married her , he'd have been asking ...
2       Sometimes he woke up in the middle of the nigh...
3       His plans and dreams had revolved around her s...
4       The best antidote for the bitterness and disap...
5       He found that if he was tired enough at night ...
6       Each day he found himself thinking less often ...
7       each day the hurt was a little duller , a litt...
8                           He had plenty of work to do .
9       The grass in the meadows came fast , now that ...
10      He could not afford to lose a drop of the prec...
11      He had no idea how much time Budd would give h...
12      He stopped every few minutes and leaned on his...
13      It could be some kind of trick Budd had though...
14      No one walked in this country , least of all E...
15      When they were closer and he saw that one was ...
16      He cleaned his shovel , left it against the fe...
17      His vi

In [104]:
sentences = X
all_words = [nltk.word_tokenize(sent) for sent in sentences]

for i in range(len(all_words)):  
    all_words[i] = [w.lower() for w in all_words[i] if w not in stopwords.words('english') and w not in string.punctuation]
    
print(all_words[:10])

[['he', 'certainly', "n't", 'want', 'wife', 'fickle', 'ann'], ['if', 'married', "'d", 'asking', 'trouble'], ['sometimes', 'woke', 'middle', 'night', 'thinking', 'ann', 'could', 'get', 'back', 'sleep'], ['his', 'plans', 'dreams', 'revolved', 'around', 'much', 'long', 'felt', 'nothing'], ['the', 'best', 'antidote', 'bitterness', 'disappointment', 'poisoned', 'hard', 'work'], ['he', 'found', 'tired', 'enough', 'night', 'went', 'sleep', 'simply', 'exhausted', 'stay', 'awake'], ['each', 'day', 'found', 'thinking', 'less', 'often', 'ann'], ['day', 'hurt', 'little', 'duller', 'little', 'less', 'poignant'], ['he', 'plenty', 'work'], ['the', 'grass', 'meadows', 'came', 'fast', 'warm', 'weather']]


In [111]:
all_words.reshape(1,1)

AttributeError: 'list' object has no attribute 'reshape'

In [105]:
X_train, X_test, y_train, y_test = train_test_split(all_words, y, test_size=0.30, random_state=42, stratify=y)

In [106]:

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer

nb = Pipeline([('tfidf', TfidfTransformer()),
               ('clf', MultinomialNB()),
              ])
nb.fit(X_train, y_train)

from sklearn.metrics import classification_report
y_pred = nb.predict(X_test)

print('accuracy %s' % accuracy_score(y_pred, y_test))
print(classification_report(y_test, y_pred,target_names=categories))

ValueError: Expected 2D array, got 1D array instead:
array=[list(['as', 'president', 'said', '``', 'arms', 'sufficient', 'beyond', 'doubt', 'certain', 'never', 'employed', '``'])
 list(['if', 'armed', 'would', "n't", 'talk', 'tough', '``'])
 list(['but', 'yet', 'real', 'estate', 'board', 'willing', 'officially', 'support', 'laws', 'admit', 'permissibility', 'introducing', 'minority', 'buyers', 'all-white', 'neighborhoods'])
 ...
 list(['by', 'counting', 'number', 'stalls', 'urinals', 'i', 'attempted', 'form', 'loose', 'estimate', 'many', 'men', 'hall', 'would', 'hold', 'one', 'time'])
 list(['christ', 'word', 'god', 'came', 'rescue', 'man', 'disobedience', 'word', 'god', 'beginning', 'brought', 'death', 'world', 'woe'])
 list(['my', 'message', 'today', '15', 'years', 'ago', 'goodness', 'people', 'strength', 'love', 'god'])].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [23]:
df.isna().sum()

Unnamed: 0    0
words         0
category      0
dtype: int64

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8200 entries, 0 to 8199
Data columns (total 3 columns):
Unnamed: 0    8200 non-null int64
words         8200 non-null object
category      8200 non-null object
dtypes: int64(1), object(2)
memory usage: 192.3+ KB


In [25]:
df.category.unique()


array(['adventure', 'science_fiction', 'religion', 'government'],
      dtype=object)

In [26]:
print(df.words[:10])

0    He certainly didn't want a wife who was fickle...
1    If he had married her , he'd have been asking ...
2    Sometimes he woke up in the middle of the nigh...
3    His plans and dreams had revolved around her s...
4    The best antidote for the bitterness and disap...
5    He found that if he was tired enough at night ...
6    Each day he found himself thinking less often ...
7    each day the hurt was a little duller , a litt...
8                        He had plenty of work to do .
9    The grass in the meadows came fast , now that ...
Name: words, dtype: object


In [28]:
X = df.words
y = df.category

In [31]:
sentences = X

all_words = [nltk.word_tokenize(sent) for sent in sentences]

for i in range(len(all_words)):  
    all_words[i] = [w.lower() for w in all_words[i] if w not in stopwords.words('english') and w not in string.punctuation]
    
print(all_words[:10])

[['he', 'certainly', "n't", 'want', 'wife', 'fickle', 'ann'], ['if', 'married', "'d", 'asking', 'trouble'], ['sometimes', 'woke', 'middle', 'night', 'thinking', 'ann', 'could', 'get', 'back', 'sleep'], ['his', 'plans', 'dreams', 'revolved', 'around', 'much', 'long', 'felt', 'nothing'], ['the', 'best', 'antidote', 'bitterness', 'disappointment', 'poisoned', 'hard', 'work'], ['he', 'found', 'tired', 'enough', 'night', 'went', 'sleep', 'simply', 'exhausted', 'stay', 'awake'], ['each', 'day', 'found', 'thinking', 'less', 'often', 'ann'], ['day', 'hurt', 'little', 'duller', 'little', 'less', 'poignant'], ['he', 'plenty', 'work'], ['the', 'grass', 'meadows', 'came', 'fast', 'warm', 'weather']]


<h3>2. Dividindo em datasets de treino e teste</h3>

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

<h3>3. Treinando o classificador</h3>

NameError: name 'CountVectorizer' is not defined

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn import neighbors

clf = neighbors.KNeighborsClassifier(n_neighbors=10, weights='uniform')

skpGrPipeline = Pipeline([ ('wrdtvec', Word2VecTransformer(1)), ('clf', clf)])
cbwPipeline = Pipeline([ ('wrdtvec', Word2VecTransformer(0)), ('clf', clf)])
glvPipeline = Pipeline([ ('glv', GloveTransformer()), ('clf', clf)])

NameError: name 'Word2VecTransformer' is not defined

In [37]:
from sklearn.model_selection import RandomizedSearchCV
import scipy

par = {'clf__n_neighbors': range(1, 100), 'clf__weights': ['uniform', 'distance']}

hyperpar_skpGr = RandomizedSearchCV(skpGrPipeline, par, cv=3, scoring='accuracy', n_jobs=2, n_iter=20)
hyperpar_cbw = RandomizedSearchCV(cbwPipeline, par, cv=3, scoring='accuracy', n_jobs=2, n_iter=20)
hyperpar_glv = RandomizedSearchCV(glvPipeline, par, cv=3, scoring='accuracy', n_jobs=2, n_iter=20)

NameError: name 'skpGrPipeline' is not defined

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer

nb = Pipeline([('vect', CountVectorizer()),
               ('tfidf', TfidfTransformer()),
               ('clf', MultinomialNB()),
              ])
nb.fit(X_train, y_train)

from sklearn.metrics import classification_report
y_pred = nb.predict(X_test)

print('accuracy %s' % accuracy_score(y_pred, y_test))
print(classification_report(y_test, y_pred,target_names=my_tags))